<a href="https://colab.research.google.com/github/niofrequency/google-collab_image/blob/main/KREA2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Krea 2 Realism

1. **Runtime → Change runtime type → L4 GPU** (A100 is fastest; a T4 works but is slow).
2. Run **cell 1** (installs + downloads; allow Google Drive access). Your LoRAs from `LORA_LINKS` are saved to
   **MyDrive/ComfyUI/loras_krea2**, and the models (10 GB or less each) to MyDrive/ComfyUI/krea2_models, so later
   sessions start faster.
3. Run **cell 2** (saves the server program) and **cell 3** (starts it — prints the `READY:` URL after the models load).
4. In the website's **Image Generator**, paste the URL into **Backend URL**. After that you can flip between
   BigLust and Krea 2 with the **Model** switch under it (it remembers each notebook's URL).

The same kinds of tools as BigLust, in their Krea 2 versions:
- **Face reference → Identity Edit V2**: your character from one face photo, restaged by the prompt (scene, pose,
  outfit, light). **Strength** = the fidelity dial: 4 recommended, 1 loose, 8+ very strict.
- **Pose / structure picture → Scene** (put your character into that picture), **Edit** (change the picture as the
  prompt says), **Depth** (copy its framing / shapes) or **Img2img**.
- **Inpaint / outpaint**, **Refiner** (4x-UltraSharp / Nomos), **LoRA targets**, **Prompt enhancer**, **Face fix**
  (follows your face photo when Face reference is on), **Eye detail**, **Hand fix**, **CodeFormer**, **Variation**.
- Not on Krea 2: PAG, Detail Daemon, Tile detail (SDXL only).

The Identity / Edit / Depth models download the first time you use them (and are saved to Google Drive).

The server runs as a separate program: if a picture runs out of memory, only the server restarts (same URL, models
reload in 1–3 minutes). **Cell 4** shows the server's log and memory use.

In [ ]:
# Cell 1: install ComfyUI and download Krea 2 Realism (+ your LoRAs)
# --- Your LoRAs: downloaded once into Google Drive (MyDrive/ComfyUI/loras_krea2) and reused every session.
#     Add more links here, or use "+ Download LoRA" on the website (those are saved to Drive too).
LORA_LINKS = [
    "https://civitai.red/api/download/models/3109006?fileId=2988982&token=777a5632707c15ecf54e13eb3b1b862b",
]
# --- Google Drive: keeps your LoRAs, and the models (<= 10 GB each) so later sessions start faster
USE_GOOGLE_DRIVE = True
def mount_google_drive():
    """Connects Google Drive. If that fails, the notebook carries on WITHOUT Drive instead of stopping
    (downloaded LoRAs then stay on this runtime only), so a Drive problem can never block you."""
    import os, shutil, time
    from google.colab import drive
    mp = "/content/drive"
    if os.path.isdir(mp) and not os.path.ismount(mp) and os.listdir(mp):
        # files left by an earlier failed run block the mount ("Mountpoint must not already contain files")
        keep = f"/content/drive_leftovers_{int(time.time())}"
        shutil.move(mp, keep)
        print(f"Moved files that were in Google Drive's way to {keep}")
    error = None
    for attempt in (1, 2):   # a second try after a clean unmount fixes most "mount failed" errors
        try:
            drive.mount(mp, force_remount=True)  # Colab asks you to allow access
            error = None
            break
        except Exception as e:
            error = e
            print(f"Google Drive try {attempt} failed ({e})" + (" — trying once more..." if attempt == 1 else ""))
            try:
                drive.flush_and_unmount()
            except Exception:
                pass
            if os.path.isdir(mp) and not os.path.ismount(mp) and os.listdir(mp):
                shutil.move(mp, f"/content/drive_leftovers_{int(time.time())}")
            time.sleep(3)
    if error is not None:
        e = error
        print(f"\n⚠️  Google Drive did not connect: {e}")
        print("    Carrying on WITHOUT Google Drive: LoRAs you download stay on this runtime only.")
        print("    To fix it next time: Runtime → Disconnect and delete runtime, run again, sign in with the SAME")
        print("    Google account as Colab, and on Google's page tick EVERY box (especially 'See, edit, create and")
        print("    delete all of your Google Drive files'). Allow pop-ups / third-party cookies for colab.research.google.com.\n")
        return False
    if not os.path.isdir(mp + "/MyDrive"):
        print("⚠️  Google Drive connected, but MyDrive is missing — carrying on without it.")
        return False
    return True
if USE_GOOGLE_DRIVE and not mount_google_drive():
    USE_GOOGLE_DRIVE = False

%cd /content
!git clone -q https://github.com/comfyanonymous/ComfyUI /content/ComfyUI || true
!cd /content/ComfyUI && git pull -q || true
!cd /content/ComfyUI && pip install -q -r requirements.txt
!pip install -q fastapi uvicorn pydantic requests psutil
# detail tools (face fix, eye / hand detail = ultralytics detectors; CodeFormer = facexlib + spandrel)
!pip install -q ultralytics facexlib spandrel spandrel_extra_arches
# Krea 2 versions of BigLust's tools (pinned versions): Identity / General Edit (face reference, scene, edit) and the
# depth control. Their models download the first time you use them on the website.
!cd /content/ComfyUI/custom_nodes && (test -d comfyui-krea2edit || git clone -q https://github.com/lbouaraba/comfyui-krea2edit)
!cd /content/ComfyUI/custom_nodes/comfyui-krea2edit && (git checkout -q 13ef60f20828949becada815d3cbc371eb5ccaab 2>/dev/null || (git fetch -q && git checkout -q 13ef60f20828949becada815d3cbc371eb5ccaab))
!cd /content/ComfyUI/custom_nodes && (test -d comfyui-krea2-controlnet || git clone -q https://github.com/facok/comfyui-krea2-controlnet)
!cd /content/ComfyUI/custom_nodes/comfyui-krea2-controlnet && (git checkout -q 79ebfd3bd80d2180b334dd7ce57f3c9ddaa0848f 2>/dev/null || (git fetch -q && git checkout -q 79ebfd3bd80d2180b334dd7ce57f3c9ddaa0848f))
!apt install -qqy aria2

import os, re, json, shutil, subprocess, threading, urllib.parse
import requests

M = "/content/ComfyUI/models"
for d in ("diffusion_models", "text_encoders", "vae", "loras", "ultralytics"):
    os.makedirs(f"{M}/{d}", exist_ok=True)

try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout.strip().split("\n")[0]
except FileNotFoundError:  # nvidia-smi only exists when a GPU is attached
    gpu = ""
if "," not in gpu:
    raise RuntimeError(
        "This Colab runtime has no GPU, and Krea 2 needs one.\n"
        "Fix: Runtime → Change runtime type → Hardware accelerator: L4 GPU (or A100) → Save, then run the cells again.\n"
        "If Colab says it can't connect to a GPU backend, your GPU time or compute units have run out for now: "
        "check Resources (top right), add compute units, or try again later.")
gpu_name, vram_mib = gpu.rsplit(",", 1)
VRAM_GIB = round(int(vram_mib.strip() or 0) / 1024, 1)
print(f"GPU: {gpu_name.strip()} ({VRAM_GIB} GB)")
if VRAM_GIB < 20:
    print("⚠️  Krea 2 is big for this GPU: it runs, but slowly (the model is stored in 8-bit). An L4 or A100 is much better.")

# --- Model files. Each one is copied to Google Drive (MyDrive/ComfyUI/krea2_models) if it's 10 GB or less,
#     so the next session copies it from Drive instead of downloading it again.
DRIVE_MODELS = "/content/drive/MyDrive/ComfyUI/krea2_models"
MAX_DRIVE_GB = 10


def get_model(url, folder, name):
    path = f"{M}/{folder}/{name}"
    if os.path.exists(path):
        return path
    cached = f"{DRIVE_MODELS}/{name}"
    if USE_GOOGLE_DRIVE and os.path.exists(cached):
        print(f"{name}: copying from Google Drive...")
        shutil.copy(cached, path + ".part")
        os.replace(path + ".part", path)
        return path
    print(f"{name}: downloading...")
    subprocess.run(["aria2c", "--console-log-level=error", "-c", "-x", "8", "-s", "8", "-k", "1M",
                    "--auto-file-renaming=false", "--allow-overwrite=true", url,
                    "-d", f"{M}/{folder}", "-o", name], check=True)
    size_gb = os.path.getsize(path) / 1e9
    if USE_GOOGLE_DRIVE and size_gb <= MAX_DRIVE_GB:
        def save():
            os.makedirs(DRIVE_MODELS, exist_ok=True)
            shutil.copy(path, cached + ".part")
            os.replace(cached + ".part", cached)
            print(f"  (saved {name} to Google Drive for next time)")
        threading.Thread(target=save, daemon=True).start()   # in the background: doesn't slow this cell down
    elif size_gb > MAX_DRIVE_GB:
        print(f"  ({name} is {size_gb:.1f} GB — over {MAX_DRIVE_GB} GB, so it's downloaded each session instead of saved to Drive)")
    return path


MODEL = get_model("https://civitai.red/api/download/models/3405129?fileId=3296411&token=777a5632707c15ecf54e13eb3b1b862b",
                  "diffusion_models", "Krea2Realism_V1.safetensors")
TE = get_model("https://huggingface.co/Comfy-Org/Krea-2/resolve/main/text_encoders/qwen3vl_4b_fp8_scaled.safetensors",
               "text_encoders", "qwen3vl_4b_fp8_scaled.safetensors")
VAE = get_model("https://huggingface.co/Comfy-Org/Krea-2/resolve/main/vae/qwen_image_vae.safetensors",
                "vae", "qwen_image_vae.safetensors")
get_model("https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8m.pt", "ultralytics", "face_yolov8m.pt")

# Smaller GPUs: keep the model's weights in 8-bit (half the memory, slightly lower quality)
model_gib = os.path.getsize(MODEL) / 2**30
weight_dtype = "fp8_e4m3fn" if model_gib > VRAM_GIB * 0.6 else "default"
json.dump({"model": os.path.basename(MODEL), "text_encoder": os.path.basename(TE), "vae": os.path.basename(VAE),
           "weight_dtype": weight_dtype, "drive_cache": DRIVE_MODELS if USE_GOOGLE_DRIVE else None},
          open("/content/krea2_settings.json", "w"))
print(f"Krea 2 model: {model_gib:.1f} GB, weights {weight_dtype}")

# --- LoRAs live in Google Drive: ComfyUI's loras folder becomes a link to MyDrive/ComfyUI/loras_krea2
LORA_LOCAL = f"{M}/loras"
if USE_GOOGLE_DRIVE:
    DRIVE_LORAS = "/content/drive/MyDrive/ComfyUI/loras_krea2"
    os.makedirs(DRIVE_LORAS, exist_ok=True)
    if not os.path.islink(LORA_LOCAL):
        for f in os.listdir(LORA_LOCAL):   # keep anything already downloaded in this session
            if not f.startswith("put_") and not os.path.exists(f"{DRIVE_LORAS}/{f}"):
                shutil.move(f"{LORA_LOCAL}/{f}", f"{DRIVE_LORAS}/{f}")
        shutil.rmtree(LORA_LOCAL)
        os.symlink(DRIVE_LORAS, LORA_LOCAL)
    print("LoRAs are stored in Google Drive (MyDrive/ComfyUI/loras_krea2)")
else:
    print("LoRAs are stored on this runtime only (lost when the runtime is deleted)")

# Download the LoRAs in LORA_LINKS (once: what's already in the folder is skipped)
index_path = f"{LORA_LOCAL}/.downloaded.json"
try:
    index = json.load(open(index_path))
except Exception:
    index = {}
for link in LORA_LINKS:
    key = re.sub(r"[?&]token=[^&]*", "", link)   # remembered without the token
    if index.get(key) and os.path.exists(f"{LORA_LOCAL}/{index[key]}"):
        print(f"LoRA already in the folder: {index[key]}")
        continue
    with requests.get(link, stream=True, timeout=60, allow_redirects=True, headers={"User-Agent": "Mozilla/5.0"}) as r:
        if r.status_code in (401, 403):
            print(f"⚠️  Civitai refused the LoRA download (needs an API key / token): {key}")
            continue
        r.raise_for_status()
        cd = r.headers.get("content-disposition", "")
        m = re.search(r"filename\*=(?:UTF-8'')?([^;]+)", cd, re.I) or re.search(r'filename="?([^";]+)"?', cd, re.I)
        name = os.path.basename(urllib.parse.unquote(m.group(1).strip())) if m else f"krea2_lora_{len(index) + 1}.safetensors"
        name = re.sub(r"[^\w.\-+() ]", "_", name)
        dest = f"{LORA_LOCAL}/{name}"
        if not os.path.exists(dest):
            print(f"Downloading LoRA {name}...")
            with open(dest + ".part", "wb") as fh:
                for chunk in r.iter_content(8 * 1024 * 1024):
                    fh.write(chunk)
            os.replace(dest + ".part", dest)
        index[key] = name
        json.dump(index, open(index_path, "w"), indent=1)
        print(f"LoRA ready: {name}")
print("LoRAs:", ", ".join(f for f in sorted(os.listdir(LORA_LOCAL)) if not f.startswith(".")) or "(none)")


In [ ]:
%%writefile /content/krea2_server.py
# Cell 2: the Krea 2 generation server, saved as its own program (/content/krea2_server.py).
# Cell 3 runs it separately from this notebook: if a picture runs out of memory only this program
# dies, and cell 3 restarts it automatically (same URL) instead of the whole Colab session crashing.
# Running this cell only saves the file; edit it here and re-run cells 2 and 3 to apply changes.
import os
import sys
import io
import re
import gc
import json
import time
import uuid
import base64
import threading
import traceback
import logging
import warnings
import urllib.parse

os.chdir("/content/ComfyUI")
sys.path.insert(0, "/content/ComfyUI")

CRASH_FILE = "/content/server_crash.txt"


def last_crash_message():
    """Set by cell 3 when the server crashed and was restarted, so the website can say why a job vanished."""
    try:
        return open(CRASH_FILE).read().strip()
    except OSError:
        return ""


print("Loading models...", flush=True)

import requests
import torch
import numpy as np
from PIL import Image

logging.getLogger("root").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

import folder_paths
import nodes
import comfy.utils
import comfy.model_management
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

# ---------------- The models (downloaded by cell 1) ----------------
SETTINGS = json.load(open("/content/krea2_settings.json"))
MODEL_NAME = SETTINGS["model"]               # the Krea 2 Realism diffusion model
CLIP_NAME = SETTINGS["text_encoder"]         # Qwen3-VL 4B text encoder
VAE_NAME = SETTINGS["vae"]
WEIGHT_DTYPE = SETTINGS.get("weight_dtype", "default")   # "fp8_e4m3fn" on smaller GPUs (cell 1 decides)

# Recommended settings (from the Krea 2 Realism workflow: few steps, CFG 1). Change them on the website.
KREA_DEFAULTS = {"steps": 8, "cfg": 1.0, "sampler": "euler", "scheduler": "simple",
                 "width": 1024, "height": 1024, "denoise": 1.0}

UNETLoader = nodes.NODE_CLASS_MAPPINGS["UNETLoader"]()
CLIPLoader = nodes.NODE_CLASS_MAPPINGS["CLIPLoader"]()
VAELoader = nodes.NODE_CLASS_MAPPINGS["VAELoader"]()
CLIPTextEncode = nodes.NODE_CLASS_MAPPINGS["CLIPTextEncode"]()
EmptyLatentImage = nodes.NODE_CLASS_MAPPINGS["EmptyLatentImage"]()
KSampler = nodes.NODE_CLASS_MAPPINGS["KSampler"]()
VAEDecode = nodes.NODE_CLASS_MAPPINGS["VAEDecode"]()
ConditioningZeroOut = nodes.NODE_CLASS_MAPPINGS["ConditioningZeroOut"]()

gc.collect()
torch.cuda.empty_cache()
with torch.inference_mode():
    unet = UNETLoader.load_unet(MODEL_NAME, WEIGHT_DTYPE)[0]
    clip = CLIPLoader.load_clip(CLIP_NAME, type="krea2")[0]
    vae = VAELoader.load_vae(VAE_NAME)[0]
print(f"Krea 2 loaded ({MODEL_NAME}, weights {WEIGHT_DTYPE}).", flush=True)

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

# ---------------- LoRAs ----------------
# Krea 2 LoRAs (not SDXL / Flux / Qwen ones). Download them from the website with "+ Download LoRA";
# they're kept in Google Drive (MyDrive/ComfyUI/loras_krea2) when it's connected.
CIVITAI_TOKEN = os.environ.get("CIVITAI_TOKEN", "")
LORA_DIR = "/content/ComfyUI/models/loras"
os.makedirs(LORA_DIR, exist_ok=True)
MODEL_EXTS = (".safetensors", ".ckpt", ".pt", ".pth", ".bin")

# ---------------- LoRA loading with a compatibility check ----------------
# ComfyUI silently skips LoRA layers that don't exist in the model, so a LoRA made for another
# model (e.g. SDXL on Qwen) would quietly do nothing. Count what matches and say so.
import comfy.lora
import comfy.lora_convert
import comfy.sd

_LORA_SUFFIXES = (".lora_up.weight", ".lora_down.weight", ".lora_A.weight", ".lora_B.weight",
                  ".lora_mid.weight", ".lora.up.weight", ".lora.down.weight", ".diff", ".hada_w1_a",
                  ".lokr_w1", ".lokr_w1_a", ".oft_blocks", ".w_norm", ".set_weight")


def apply_lora_checked(model, clip, name, strength_model, strength_clip, model_label):
    """Apply a LoRA like ComfyUI's LoraLoader. Returns (model, clip, warning_or_None).
    Raises ValueError if none of the LoRA's layers match the model."""
    path = folder_paths.get_full_path_or_raise("loras", name)
    lora, metadata = comfy.utils.load_torch_file(path, safe_load=True, return_metadata=True)
    key_map = {}
    if model is not None:
        key_map = comfy.lora.model_lora_keys_unet(model.model, key_map)
    if clip is not None:
        key_map = comfy.lora.model_lora_keys_clip(clip.cond_stage_model, key_map)
    converted = comfy.lora_convert.convert_lora(dict(lora))
    patches = comfy.lora.load_lora(converted, key_map, log_missing=False)
    matched = sum(1 for k in patches if not (isinstance(k, str) and k.endswith(".config")))
    layers = {k[: -len(s)] for k in converted for s in _LORA_SUFFIXES if k.endswith(s)}
    total = len(layers)
    if matched == 0:
        raise ValueError(
            f"LoRA '{name}' doesn't match the {model_label} model (0 of {total or '?'} layers matched). "
            f"It was probably made for a different model — remove it or pick a {model_label} LoRA.")
    warning = None
    if total and matched < 0.5 * total:
        warning = (f"LoRA '{name}': only {matched} of {total} layers matched the {model_label} model, "
                   f"so it may have little effect (probably made for a different model version).")
    print(f"  LoRA {name} @ {strength_model}: {matched}/{total or '?'} layers matched")
    new_model, new_clip = comfy.sd.load_lora_for_models(model, clip, lora, strength_model, strength_clip,
                                                        lora_metadata=metadata)
    return new_model, new_clip, warning


def list_files(folder):
    out = []
    for root, dirs, files in os.walk(folder):
        # skip training save points (e.g. a copied output folder: <name>-step00000200-state/model.safetensors)
        dirs[:] = [d for d in dirs if not d.endswith("-state") and not d.startswith(".")]
        for f in files:
            if f.lower().endswith(MODEL_EXTS) and f.lower() not in ("optimizer.bin", "scheduler.bin", "model.safetensors"):
                out.append(os.path.relpath(os.path.join(root, f), folder))
    return sorted(out, key=str.lower)




_lora_model_cache = {}   # (LoRA combination) -> (model, warnings); the last 3 are kept


def wanted_loras(loras, target):
    """The (name, strength) of the LoRAs for "main" (main picture) or "face" (face fix)."""
    out = []
    for l in loras or []:
        name = (l.get("name") if isinstance(l, dict) else str(l)) or ""
        strength = float(l.get("strength", 1.0)) if isinstance(l, dict) else 1.0
        where = (l.get("target") if isinstance(l, dict) else None) or "all"
        if name and strength != 0 and where in ("all", target):
            out.append((name, strength))
    return out


def model_with_loras(wanted, extra=()):
    """The Krea 2 model with these LoRAs (+ tool LoRAs such as Identity Edit) applied. Returns (model, warnings)."""
    key = tuple(wanted) + tuple(extra)
    if not key:
        return unet, []
    if key in _lora_model_cache:
        return _lora_model_cache[key]
    available = set(list_files(LORA_DIR))
    model, warns = unet, []
    with torch.inference_mode():
        for name, strength in list(wanted) + list(extra):
            if name not in available and not os.path.exists(os.path.join(TOOLS_DIR, name)):
                raise ValueError(f"LoRA not found on the server: {name}")
            model, _, warning = apply_lora_checked(model, None, name, strength, 0, "Krea 2")
            if warning:
                warns.append(warning)
    while len(_lora_model_cache) >= 3:
        _lora_model_cache.pop(next(iter(_lora_model_cache)))
    _lora_model_cache[key] = (model, warns)
    return model, warns


# ---------------- LoRA downloads (background job, streamed to disk) ----------------
def resolve_url(url):
    """Turn page links into direct download links. Returns (download_url, suggested_name).
    Civitai: civitai.com and civitai.red links both work (the host you pasted is kept)."""
    url = url.strip()
    parsed = urllib.parse.urlparse(url)
    qs = urllib.parse.parse_qs(parsed.query)
    token = (qs.get("token") or [CIVITAI_TOKEN])[0]
    host = parsed.netloc.lower()
    if "civitai.com" in host or "civitai.red" in host:
        base = f"https://{host}"
        m = re.search(r"/api/download/models/(\d+)", parsed.path)
        if m:
            version_id = m.group(1)
        else:
            version_id = (qs.get("modelVersionId") or [None])[0]
            if not version_id:
                mm = re.search(r"/models/(\d+)", parsed.path)
                if not mm:
                    raise ValueError("Unrecognised Civitai link. Use a model page or download link.")
                headers = {"Authorization": f"Bearer {token}"} if token else {}
                info = requests.get(f"{base}/api/v1/models/{mm.group(1)}", headers=headers, timeout=30)
                info.raise_for_status()
                versions = info.json().get("modelVersions") or []
                if not versions:
                    raise ValueError("That Civitai model has no downloadable versions.")
                version_id = versions[0]["id"]  # newest version
        keep = {k: v[0] for k, v in qs.items() if k in ("fileId", "type", "format")}
        if token:
            keep["token"] = token
        return f"{base}/api/download/models/{version_id}" + ("?" + urllib.parse.urlencode(keep) if keep else ""), None
    if "huggingface.co" in host:
        url = url.replace("/blob/", "/resolve/")
        return url, os.path.basename(urllib.parse.urlparse(url).path)
    return url, os.path.basename(parsed.path)


def filename_from_response(resp, fallback):
    cd = resp.headers.get("content-disposition", "")
    m = re.search(r"filename\*=(?:UTF-8'')?([^;]+)", cd, re.I) or re.search(r'filename="?([^";]+)"?', cd, re.I)
    name = urllib.parse.unquote(m.group(1).strip()) if m else (fallback or "")
    name = os.path.basename(name).strip().strip('"')
    name = re.sub(r"[^\w.\-+() ]", "_", name)
    return name


def run_download(job_id, url, kind):
    job = JOBS[job_id]
    job["status"] = "running"
    folder = LORA_DIR
    tmp_path = None
    try:
        dl_url, suggested = resolve_url(url)
        with requests.get(dl_url, stream=True, timeout=60, allow_redirects=True,
                          headers={"User-Agent": "Mozilla/5.0"}) as r:
            if r.status_code in (401, 403):
                raise ValueError("Access denied — this model needs a Civitai API key (set CIVITAI_TOKEN or add ?token=...)")
            r.raise_for_status()
            if "text/html" in r.headers.get("content-type", ""):
                raise ValueError("The link returned a web page, not a model file. Use the download link.")
            name = filename_from_response(r, suggested)
            if not name.lower().endswith(MODEL_EXTS):
                name = f"{kind}_{int(time.time())}.safetensors"
            job["filename"] = name
            path = os.path.join(folder, name)
            if os.path.exists(path):
                job["status"], job["progress"] = "done", 100
                return
            total = int(r.headers.get("content-length") or 0)
            job["total_mb"] = round(total / 1e6) if total else None
            tmp_path = path + ".part"
            done = 0
            with open(tmp_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                    f.write(chunk)
                    done += len(chunk)
                    job["done_mb"] = round(done / 1e6)
                    job["progress"] = round(done * 100 / total) if total else None
            os.replace(tmp_path, path)
            tmp_path = None
        job["status"], job["progress"] = "done", 100
        print(f"Downloaded {kind}: {name} ({job.get('done_mb')} MB)")
    except Exception as e:
        traceback.print_exc()
        job["status"], job["error"] = "error", str(e)
    finally:
        if tmp_path and os.path.exists(tmp_path):
            os.remove(tmp_path)



class GenerateRequest(BaseModel):
    positive: str = ""
    negative: str = ""
    steps: int = KREA_DEFAULTS["steps"]
    cfg: float = KREA_DEFAULTS["cfg"]
    width: int = KREA_DEFAULTS["width"]
    height: int = KREA_DEFAULTS["height"]
    seed: int = -1
    sampler: str = KREA_DEFAULTS["sampler"]
    scheduler: str = KREA_DEFAULTS["scheduler"]
    denoise: float = 1.0
    batch_size: int = 1
    checkpoint: str | None = None
    loras: list = []
    # detail tools (same names as BigLust's, so the website's Detail tools box works for both)
    face_fix: bool = False
    face_fix_strength: float = 0.4
    face_fix_passes: int = 1
    eye_fix: bool = False
    eye_fix_strength: float = 0.3
    hand_fix: bool = False
    hand_fix_strength: float = 0.4
    face_restore: bool = False
    face_restore_fidelity: float = 0.7
    face_restore_blend: float = 0.4
    variation_seed: int = -1
    variation_strength: float = 0.0
    # LoRA targets: each LoRA can be "all", "main" (main picture only) or "face" (face fix only)
    # Face reference = Krea 2 Identity Edit V2: your character from one photo, restaged by the prompt
    face_ref_images: list = []
    face_ref_method: str = "identity"
    face_ref_strength: float = 4.0        # the fidelity dial: 1 = loose · 4 = recommended · 8+ = very strict
    # Second picture: "scene" (put the person into this picture), "edit" (change this picture as the prompt says),
    # "kdepth" (copy its framing / shapes with the Krea 2 depth control) or "img2img"
    ref_image: str | None = None
    ref_mode: str = "scene"
    ref_strength: float = 1.0
    ref_end_percent: float = 1.0
    ref_denoise: float = 0.6
    # Inpaint / outpaint: only where mask_image is white changes; the rest is pasted back from init_image
    init_image: str | None = None
    inpaint_denoise: float = 1.0
    mask_grow: int = 8
    mask_feather: int = 16
    # Refiner: enlarge, then lightly redraw with Krea 2
    refiner: bool = False
    refiner_checkpoint: str | None = None
    refiner_strength: float = 0.3
    refiner_upscale: float = 1.5
    refiner_steps: int = 8
    refiner_upscaler: str = "lanczos"
    reference_image: str | None = None    # Qwen-style fields (not used by Krea 2)
    reference_images: list = []
    mask_image: str | None = None


def pil_to_b64(img):
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")


MAX_SEED = 2**32 - 1   # also an exact number on the website, so a seed can be reused
MAX_SIDE = 2048


def size16(v):
    return max(256, min(int(v or 1024), MAX_SIDE)) // 16 * 16


_COND_CACHE = {}
MODELS_DIR = "/content/ComfyUI/models"
VAEEncode = nodes.NODE_CLASS_MAPPINGS["VAEEncode"]()


def encode(text):
    """Prompt -> conditioning, cached (the same prompt again skips the text encoder)."""
    if text not in _COND_CACHE:
        if len(_COND_CACHE) >= 4:
            _COND_CACHE.pop(next(iter(_COND_CACHE)))
        with torch.inference_mode():
            _COND_CACHE[text] = CLIPTextEncode.encode(clip, text)[0]
    return _COND_CACHE[text]

def tensor_to_pil(img_tensor):
    return Image.fromarray((img_tensor.float().cpu().numpy().clip(0, 1) * 255).round().astype(np.uint8))


# ---------------- Detail tools (the same as BigLust's, with Krea 2 doing the redrawing) ----------------
# Face fix (like ADetailer): each face is cut out with some room around it, redrawn at ~1024 px with the same
# model, LoRAs and prompt (only partly: the strength), and blended back in. Small faces get far more detail.
# Eye detail / hand fix work the same way on the eye area / hands. CodeFormer cleans up faces at the very end.
FACE_MODEL_PATH = os.path.join(MODELS_DIR, "ultralytics", "face_yolov8m.pt")
HAND_MODEL = ("Bingsu/adetailer", "hand_yolov8s.pt")
_detectors = {}


def yolo(path):
    if path not in _detectors:
        from ultralytics import YOLO
        _detectors[path] = YOLO(path)
    return _detectors[path]


def detail_tools_available():
    try:
        import ultralytics  # noqa: F401
        return os.path.exists(FACE_MODEL_PATH)
    except Exception:
        return False


def _boxes(results, limit):
    if not results or results[0].boxes is None:
        return []
    boxes = results[0].boxes.xyxy.cpu().numpy().tolist()
    boxes.sort(key=lambda b: (b[2] - b[0]) * (b[3] - b[1]), reverse=True)
    return boxes[:limit]


def detect_faces(pil_img, conf=0.35, max_faces=4):
    """Face boxes (x1, y1, x2, y2), biggest first. Runs on the CPU so it takes no GPU memory."""
    return _boxes(yolo(FACE_MODEL_PATH).predict(pil_img, conf=conf, verbose=False, device="cpu"), max_faces)


def detect_hands(pil_img, conf=0.35, max_hands=4):
    path = os.path.join(MODELS_DIR, "ultralytics", HAND_MODEL[1])
    if not os.path.exists(path):   # ~25 MB, downloaded the first time hand fix is used
        from huggingface_hub import hf_hub_download
        os.makedirs(os.path.dirname(path), exist_ok=True)
        os.replace(hf_hub_download(*HAND_MODEL, local_dir=os.path.join(MODELS_DIR, "ultralytics", ".hf")), path)
    return _boxes(yolo(path).predict(pil_img, conf=conf, verbose=False, device="cpu"), max_hands)


def eye_boxes(face_boxes):
    """The eye band of each face box (upper-middle part of the face)."""
    return [(x1 + 0.12 * (x2 - x1), y1 + 0.22 * (y2 - y1), x2 - 0.12 * (x2 - x1), y1 + 0.58 * (y2 - y1))
            for x1, y1, x2, y2 in face_boxes if x2 - x1 >= 40]


def redraw_regions(pil_img, boxes, model, positive, negative, seed, req, strength, context=1.8, prep=None):
    """Each box (with some room around it) is cut out, redrawn at ~1024 px and blended back with soft edges.
    Returns (image, regions redrawn)."""
    from PIL import ImageDraw, ImageFilter
    out = pil_img.copy()
    W, H = out.size
    fixed = 0
    for x1, y1, x2, y2 in boxes:
        bw, bh = x2 - x1, y2 - y1
        if bw < 8 or bh < 8:
            continue
        side = int(min(max(bw, bh) * context, W, H))
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        left = int(max(0, min(cx - side / 2, W - side)))
        top = int(max(0, min(cy - side / 2, H - side)))
        crop = out.crop((left, top, left + side, top + side))
        work_size = 1024 if side < 1024 else (side // 16) * 16
        work = crop.resize((work_size, work_size), Image.Resampling.LANCZOS)
        m, pos, neg = prep(work_size) if prep else (model, positive, negative)
        with torch.inference_mode():
            latent = VAEEncode.encode(vae, torch.from_numpy(np.array(work).astype(np.float32) / 255.0)[None, ])[0]
            samples = KSampler.sample(m, seed, max(1, int(req.steps)), float(req.cfg), req.sampler, req.scheduler,
                                      pos, neg, latent, strength)[0]
            redone = tensor_to_pil(VAEDecode.decode(vae, samples)[0][0]).resize((side, side), Image.Resampling.LANCZOS)
        feather = max(4, side // 10)   # soft edge, so no square shows
        mask = Image.new("L", (side, side), 0)
        ImageDraw.Draw(mask).rectangle((feather, feather, side - feather, side - feather), fill=255)
        out.paste(redone, (left, top), mask.filter(ImageFilter.GaussianBlur(feather / 2)))
        fixed += 1
    return out, fixed


CODEFORMER = ("codeformer.pth", "https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/codeformer.pth")
_restore = {}


def face_restore_available():
    try:
        import facexlib  # noqa: F401
        import spandrel_extra_arches  # noqa: F401
        return True
    except Exception:
        return False


def restore_faces(pil, fidelity=0.7, blend=0.4, job=None):
    """CodeFormer on every face (fidelity 0 = strongest cleanup, 1 = closest to the face), blended in partly.
    Returns (image, faces restored)."""
    import cv2
    from facexlib.utils.face_restoration_helper import FaceRestoreHelper
    dev = comfy.model_management.get_torch_device()
    if "model" not in _restore:
        folder = os.path.join(MODELS_DIR, "facerestore_models")
        path = os.path.join(folder, CODEFORMER[0])
        if not os.path.exists(path):
            if job is not None:
                job["phase"] = "downloading CodeFormer (once, ~0.4 GB)"
            os.makedirs(folder, exist_ok=True)
            import urllib.request
            urllib.request.urlretrieve(CODEFORMER[1], path + ".part")
            os.replace(path + ".part", path)
        import spandrel_extra_arches
        from spandrel import ModelLoader
        spandrel_extra_arches.install()
        _restore["model"] = ModelLoader().load_from_file(path).eval()
        _restore["helper"] = FaceRestoreHelper(1, face_size=512, crop_ratio=(1, 1), det_model="retinaface_resnet50",
                                               save_ext="png", use_parse=True, device=dev,
                                               model_rootpath=os.path.join(MODELS_DIR, "facexlib"))
    net, helper = _restore["model"], _restore["helper"]
    net.to(dev)
    helper.clean_all()
    bgr = cv2.cvtColor(np.array(pil.convert("RGB")), cv2.COLOR_RGB2BGR)
    helper.read_image(bgr)
    helper.get_face_landmarks_5(only_center_face=False, resize=640, eye_dist_threshold=5)
    helper.align_warp_face()
    if not helper.cropped_faces:
        net.to("cpu")
        return pil, 0
    with torch.no_grad():
        for face in helper.cropped_faces:
            t = torch.from_numpy(cv2.cvtColor(face, cv2.COLOR_BGR2RGB)).permute(2, 0, 1).float().div(255)[None].to(dev)
            o = net.model(t * 2 - 1, weight=float(fidelity))[0]          # CodeFormer works in -1..1
            o = ((o.float().clamp(-1, 1) + 1) / 2)[0].permute(1, 2, 0).mul(255).round().byte().cpu().numpy()
            helper.add_restored_face(cv2.addWeighted(cv2.cvtColor(o, cv2.COLOR_RGB2BGR), blend, face, 1 - blend, 0))
    helper.get_inverse_affine(None)
    out = helper.paste_faces_to_input_image(upsample_img=bgr)
    n = len(helper.cropped_faces)
    net.to("cpu")
    torch.cuda.empty_cache()
    return Image.fromarray(cv2.cvtColor(out.astype(np.uint8), cv2.COLOR_BGR2RGB)), n


# ---------------- Variation seed ----------------
# Same picture, slightly different: the main seed's starting noise is blended (spherically) with a second seed's.
import contextlib


def slerp_noise(a, b, t):
    af, bf = a.flatten().float(), b.flatten().float()
    dot = ((af / af.norm()) * (bf / bf.norm())).sum().clamp(-1.0, 1.0)
    if dot.abs() > 0.9995:
        out = a.float() * (1 - t) + b.float() * t
    else:
        omega = torch.acos(dot)
        so = torch.sin(omega)
        out = (torch.sin((1 - t) * omega) / so) * a.float() + (torch.sin(t * omega) / so) * b.float()
    return out.to(a.dtype)


@contextlib.contextmanager
def variation_noise(var_seed, strength):
    import comfy.sample
    if var_seed is None or not strength or strength <= 0:
        yield
        return
    original = comfy.sample.prepare_noise

    def blended(latent_image, seed, noise_inds=None):
        base = original(latent_image, seed, noise_inds)
        if not isinstance(base, torch.Tensor):
            return base
        return slerp_noise(base, original(latent_image, var_seed, noise_inds), float(strength))
    comfy.sample.prepare_noise = blended
    try:
        yield
    finally:
        comfy.sample.prepare_noise = original


# ================= Krea 2 versions of BigLust's tools =================
# Face reference -> Krea 2 Identity Edit V2 (lbouaraba/comfyui-krea2edit): the face photo goes in as reference tokens
#   AND is shown to the text encoder, and the Identity Edit LoRA restages that person as the prompt says.
# Pose / structure picture -> "scene" (put the person into this picture), "edit" (General Edit V2: change the
#   picture as the prompt says), "kdepth" (Krea 2 depth Control LoRA, facok/comfyui-krea2-controlnet) or img2img.
# The tool models download the first time they're used (and are kept in Google Drive if it's connected).
import importlib.util
import shutil

TOOLS_DIR = "/content/ComfyUI/models/krea2_tools"
os.makedirs(TOOLS_DIR, exist_ok=True)
folder_paths.add_model_folder_path("loras", TOOLS_DIR)   # found by name, but not listed with your LoRAs
DRIVE_CACHE = SETTINGS.get("drive_cache")                 # MyDrive/ComfyUI/krea2_models, or None
PACKS = {"edit": "/content/ComfyUI/custom_nodes/comfyui-krea2edit",
         "control": "/content/ComfyUI/custom_nodes/comfyui-krea2-controlnet"}
TOOL_FILES = {"identity": ("conradlocke/krea2-identity-edit", "krea2_identity_edit_v2.safetensors"),
              "edit": ("conradlocke/krea2-identity-edit", "krea2_general_edit_v2.safetensors"),
              "depth": ("Patil/Krea-2-depth-controlnet", None)}
REF_MODES = ("scene", "edit", "kdepth", "img2img")
_packs = {}


def packs_available():
    return all(os.path.isfile(os.path.join(p, "__init__.py")) for p in PACKS.values())


def pack(kind):
    """The node pack's nodes (loaded once; custom nodes aren't auto-loaded when ComfyUI is used as a library)."""
    if kind not in _packs:
        path = PACKS[kind]
        if not os.path.isfile(os.path.join(path, "__init__.py")):
            raise ValueError("The Krea 2 tools aren't installed in this notebook — open it fresh from GitHub and run cell 1 again.")
        name = f"krea2pack_{kind}"
        spec = importlib.util.spec_from_file_location(name, os.path.join(path, "__init__.py"), submodule_search_locations=[path])
        mod = importlib.util.module_from_spec(spec)
        sys.modules[name] = mod
        spec.loader.exec_module(mod)
        _packs[kind] = mod.NODE_CLASS_MAPPINGS
    return _packs[kind]


def tool_file(kind, job=None):
    """Path of a tool model; downloaded once from Hugging Face (or copied from Google Drive)."""
    repo, want = TOOL_FILES[kind]
    local = os.path.join(TOOLS_DIR, want or "krea2_depth_control.safetensors")
    if os.path.exists(local):
        return local
    name = os.path.basename(local)
    if DRIVE_CACHE and os.path.exists(os.path.join(DRIVE_CACHE, name)):
        if job is not None:
            job["phase"] = f"copying {name} from Google Drive"
        shutil.copy(os.path.join(DRIVE_CACHE, name), local + ".part")
        os.replace(local + ".part", local)
        return local
    from huggingface_hub import hf_hub_download, list_repo_files
    files = [f for f in list_repo_files(repo) if f.endswith(".safetensors")]
    if want:
        pick = next((f for f in files if os.path.basename(f) == want), None)
    else:   # the depth control LoRA: the file with "depth" in its name, else the only one
        pick = next((f for f in files if "depth" in f.lower()), files[0] if len(files) == 1 else None)
    if not pick:
        raise ValueError(f"Couldn't find {want or 'the depth control model'} in {repo} on Hugging Face.")
    if job is not None:
        job["phase"] = f"downloading {os.path.basename(pick)} (once)"
    print(f"Downloading {repo}/{pick}...", flush=True)
    src = hf_hub_download(repo, pick, local_dir=os.path.join(TOOLS_DIR, ".hf"))
    os.replace(src, local)
    if DRIVE_CACHE and os.path.getsize(local) <= 10e9:
        def save():
            try:
                os.makedirs(DRIVE_CACHE, exist_ok=True)
                shutil.copy(local, os.path.join(DRIVE_CACHE, name + ".part"))
                os.replace(os.path.join(DRIVE_CACHE, name + ".part"), os.path.join(DRIVE_CACHE, name))
            except Exception as e:
                print(f"Couldn't save {name} to Google Drive: {e}", flush=True)
        threading.Thread(target=save, daemon=True).start()
    if job is not None:
        job["phase"] = None
    return local


def decode_b64_image(data):
    if not data:
        return None
    if data.strip().startswith("data:") and "," in data:
        data = data.split(",", 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(data))).convert("RGB")


def pil_to_tensor(pil):
    return torch.from_numpy(np.array(pil.convert("RGB")).astype(np.float32) / 255.0)[None, ]


def fit_cover(pil, w, h):
    """Resize to cover w x h and crop the middle (no stretching)."""
    scale = max(w / pil.width, h / pil.height)
    nw, nh = max(w, round(pil.width * scale)), max(h, round(pil.height * scale))
    pil = pil.resize((nw, nh), Image.Resampling.LANCZOS)
    left, top = (nw - w) // 2, (nh - h) // 2
    return pil.crop((left, top, left + w, top + h))


def empty_latent(w, h):
    return {"samples": torch.zeros([1, 16, h // 8, w // 8])}


def identity_model(model, refs, w, h, boost, positions):
    """Krea2 Edit source patch: refs (1 or 2 pictures: [person] or [scene, person]) become reference tokens."""
    patch = pack("edit")["Krea2EditModelPatch"]()
    imgs = [pil_to_tensor(p) for p in refs]
    with torch.inference_mode():
        first = VAEEncode.encode(vae, pil_to_tensor(refs[0].resize((256, 256))))[0]   # required input; the pixel path is used
        kw = dict(vae=vae, source_image=imgs[0], target_latent=empty_latent(w, h), fit_mode="fit")
        if len(imgs) == 2:
            second = VAEEncode.encode(vae, pil_to_tensor(refs[1].resize((256, 256))))[0]
            kw.update(source_latent_b=second, source_image_b=imgs[1], ref_boost=float(boost), ref_boost_a=1.0,
                      ref_position=positions[1], ref_position_a=positions[0])
        else:
            kw.update(ref_boost=float(boost), ref_position=positions[0])
        return patch.patch(model, first, **kw)[0]


def grounded(text, refs, px=768):
    """The prompt encoded while the text encoder looks at the reference pictures (how the Edit LoRAs were trained)."""
    enc = pack("edit")["Krea2EditGroundedEncode"]()
    imgs = [pil_to_tensor(p) for p in refs]
    with torch.inference_mode():
        return enc.encode(clip, text, image=imgs[0], image_b=imgs[1] if len(imgs) > 1 else None, grounding_px=px)[0]


_depth = {}


def depth_map(pil, w, h):
    """Depth Anything V2 Small (on the CPU: no GPU memory) -> a depth picture at the output size."""
    if "pipe" not in _depth:
        from transformers import pipeline
        _depth["pipe"] = pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf", device="cpu")
    out = _depth["pipe"](fit_cover(pil, w, h))["depth"].convert("RGB")
    return out if out.size == (w, h) else out.resize((w, h), Image.Resampling.LANCZOS)


def depth_model(model, pil, w, h, strength, job=None):
    """Krea 2 depth Control LoRA following the picture's depth map. Returns (model, depth picture)."""
    nodes_ = pack("control")
    path = tool_file("depth", job)
    dm = depth_map(pil, w, h)
    with torch.inference_mode():
        lat, _ = nodes_["Krea2ControlImageEncode"]().encode(pil_to_tensor(dm), vae, "match_latent_size", "lanczos", "center",
                                                            "grayscale", "per_image_minmax", False, "independent_images",
                                                            empty_latent(w, h))
        m = nodes_["Krea2ControlLoRALoader"]().load_lora(model, os.path.basename(path), float(strength))[0]
        return nodes_["Krea2ControlApply"]().apply(m, lat)[0], dm


# ---------------- Inpaint / outpaint ----------------
def prepare_mask(mask_pil, size, grow):
    """Black/white mask at the picture's size, widened by `grow` px."""
    from PIL import ImageFilter
    m = mask_pil.convert("L").resize(size, Image.Resampling.NEAREST).point(lambda v: 255 if v > 127 else 0)
    if grow > 0:
        m = m.filter(ImageFilter.GaussianBlur(grow / 2)).point(lambda v: 255 if v > 8 else 0)
    return m


def paste_back(result, original, mask, feather):
    """Only the masked area comes from the result; everything else is the original, pixel for pixel."""
    from PIL import ImageFilter
    if result.size != original.size:
        result = result.resize(original.size, Image.Resampling.LANCZOS)
    soft = mask.filter(ImageFilter.GaussianBlur(feather / 2)) if feather > 0 else mask
    soft = Image.fromarray(np.minimum(np.array(soft), np.array(mask)))
    return Image.composite(result.convert("RGB"), original.convert("RGB"), soft)


# ---------------- Refiner: enlarge (simple resize or an upscale model), then lightly redraw ----------------
UPSCALERS = {
    "ultrasharp": ("4x-UltraSharp.pth", "https://huggingface.co/lokCX/4x-Ultrasharp/resolve/main/4x-UltraSharp.pth", 512),
    "nomos8kdat": ("4xNomos8kDAT.pth", "https://github.com/Phhofm/models/releases/download/4xNomos8kDAT/4xNomos8kDAT.pth", 256),
}
_upscalers = {}


def upscalers_available():
    try:
        import spandrel  # noqa: F401
        return list(UPSCALERS)
    except Exception:
        return []


def model_upscale(pil, key, size, job=None):
    """pil -> size (w, h), enlarged with the upscale model in overlapping tiles."""
    if key not in _upscalers:
        name, url, _ = UPSCALERS[key]
        path = os.path.join(MODELS_DIR, "upscale_models", name)
        if not os.path.exists(path):
            if job is not None:
                job["phase"] = f"downloading {name} (once)"
            import urllib.request
            os.makedirs(os.path.dirname(path), exist_ok=True)
            urllib.request.urlretrieve(url, path + ".part")
            os.replace(path + ".part", path)
        from spandrel import ModelLoader
        _upscalers[key] = ModelLoader().load_from_file(path).eval()
    m = _upscalers[key]
    tile, pad = UPSCALERS[key][2], 16
    dev = comfy.model_management.get_torch_device()
    half = dev.type == "cuda" and m.supports_half
    m.to(dev)
    if half:
        m.half()
    try:
        img = torch.from_numpy(np.array(pil.convert("RGB"))).permute(2, 0, 1).float().div(255)[None].to(dev)
        if half:
            img = img.half()
        _, _, H, W = img.shape
        sc = m.scale
        out = torch.zeros((1, 3, H * sc, W * sc), dtype=img.dtype, device="cpu")
        with torch.no_grad():
            for y in range(0, H, tile):
                for x in range(0, W, tile):
                    th, tw = min(tile, H - y), min(tile, W - x)
                    y0, x0 = max(0, y - pad), max(0, x - pad)
                    y1, x1 = min(H, y + th + pad), min(W, x + tw + pad)
                    o = m(img[:, :, y0:y1, x0:x1])
                    out[:, :, y * sc:(y + th) * sc, x * sc:(x + tw) * sc] = \
                        o[:, :, (y - y0) * sc:(y - y0 + th) * sc, (x - x0) * sc:(x - x0 + tw) * sc].cpu()
        big = Image.fromarray(out[0].float().clamp(0, 1).permute(1, 2, 0).mul(255).round().byte().numpy())
    finally:
        m.to("cpu")
        if half:
            m.float()
        torch.cuda.empty_cache()
    return big if big.size == tuple(size) else big.resize(tuple(size), Image.Resampling.LANCZOS)


TILED_DECODE_PIXELS = 1_600_000   # bigger pictures are decoded in tiles: much less GPU memory, same result


def decode_image(samples):
    lat = samples.get("samples")
    pixels = lat.shape[-1] * 8 * lat.shape[-2] * 8 if hasattr(lat, "shape") else 0
    with torch.inference_mode():
        if pixels <= TILED_DECODE_PIXELS:
            try:
                return VAEDecode.decode(vae, samples)[0]
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
        node = nodes.NODE_CLASS_MAPPINGS["VAEDecodeTiled"]()
        try:
            return node.decode(vae, samples, 512, 64, 64, 8)[0]
        except TypeError:
            return node.decode(vae, samples, 512, 64)[0]


def refine(pil, req, model, positive, negative, seed, job):
    factor = max(1.0, min(float(req.refiner_upscale or 1.0), 2.0))
    strength = max(0.05, min(float(req.refiner_strength or 0.3), 0.8))
    steps = max(4, min(int(req.refiner_steps or 8), 60))
    scale = min(factor, 2048 / max(pil.width, pil.height))
    w, h = int(round(pil.width * scale / 16)) * 16, int(round(pil.height * scale / 16)) * 16
    up = (req.refiner_upscaler or "lanczos").lower()
    if up in UPSCALERS and (w, h) != pil.size and up in upscalers_available():
        job["phase"] = f"enlarging with {UPSCALERS[up][0].rsplit('.', 1)[0]}"
        big = model_upscale(pil, up, (w, h), job)
    else:
        big = pil.resize((w, h), Image.Resampling.LANCZOS) if (w, h) != pil.size else pil
    job["phase"] = "refining"
    with torch.inference_mode():
        latent = VAEEncode.encode(vae, pil_to_tensor(big))[0]
        samples = KSampler.sample(model, seed, steps, float(req.cfg), req.sampler, req.scheduler, positive, negative, latent, strength)[0]
    return tensor_to_pil(decode_image(samples)[0])


# ---------------- LoRA tags (the website's prompt enhancer) ----------------
# kohya-trained LoRAs store how often each caption tag was used ("ss_tag_frequency") in the file's small header.
_lora_tags_cache = {}


def read_lora_tags(path):
    import struct
    with open(path, "rb") as fh:
        n = struct.unpack("<Q", fh.read(8))[0]
        if n > 100_000_000:
            raise ValueError("not a .safetensors file")
        meta = (json.loads(fh.read(n)) or {}).get("__metadata__") or {}

    def as_json(key):
        try:
            return json.loads(meta.get(key) or "{}")
        except Exception:
            return {}
    counts, spelling = {}, {}
    for tags in as_json("ss_tag_frequency").values():
        for tag, count in (tags or {}).items():
            tag = str(tag).strip()
            if not tag:
                continue
            key = tag.lower()
            spelling.setdefault(key, tag)
            counts[key] = counts.get(key, 0) + int(count or 0)
    images = sum(int((d or {}).get("img_count") or 0) for d in as_json("ss_dataset_dirs").values())
    if not images:
        try:
            images = int(meta.get("ss_num_train_images") or 0)
        except ValueError:
            images = 0
    images = max(images, max(counts.values(), default=0))
    trigger = [t.strip() for t in re.split(r"[,\n]", meta.get("modelspec.trigger_phrase") or meta.get("ss_trigger_words") or "") if t.strip()]
    tags = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))[:400]
    return {"images": images, "trigger": trigger, "tags": [{"tag": spelling[k], "count": c} for k, c in tags],
            "base_model": meta.get("ss_sd_model_name") or meta.get("modelspec.architecture") or None}


@app.get("/lora_tags")
def lora_tags(name: str):
    path = os.path.realpath(os.path.join(LORA_DIR, name))
    root = os.path.realpath(LORA_DIR)
    if not (path == root or path.startswith(root + os.sep)) or not os.path.isfile(path):
        return {"error": f"LoRA not found on the server: {name}"}
    if not path.lower().endswith(".safetensors"):
        return {"name": name, "images": 0, "trigger": [], "tags": [], "note": "Only .safetensors LoRAs carry their training tags"}
    key = (path, os.path.getmtime(path))
    if key not in _lora_tags_cache:
        try:
            _lora_tags_cache[key] = read_lora_tags(path)
        except Exception as e:
            return {"error": f"Could not read {name}: {e}"}
    out = {"name": name, **_lora_tags_cache[key]}
    if not out["tags"]:
        out["note"] = "This LoRA has no training tags saved in it (made without kohya, or with metadata turned off)"
    return out


# Generation runs in a background job. Cloudflare quick tunnels drop requests that take
# longer than ~100 s, which a T4 easily exceeds, so the frontend polls /job/{id} instead.
# Kept across re-runs of this cell, so jobs that are running keep their state and never run two at once
JOBS = globals().get("JOBS") if isinstance(globals().get("JOBS"), dict) else {}
gpu_lock = globals().get("gpu_lock") if isinstance(globals().get("gpu_lock"), type(threading.Lock())) else threading.Lock()
current_job = globals().get("current_job") if isinstance(globals().get("current_job"), dict) else {"id": None}


def progress_hook(value, total, *args, **kwargs):
    job = JOBS.get(current_job["id"])
    if job and total:
        job["step"], job["total_steps"] = int(value), int(total)


comfy.utils.set_progress_bar_global_hook(progress_hook)


def new_job(kind, **extra):
    now = time.time()
    for jid in [j for j, v in JOBS.items() if v["status"] in ("done", "error") and now - v["created"] > 1800]:
        JOBS.pop(jid, None)
    job_id = uuid.uuid4().hex[:12]
    JOBS[job_id] = {"kind": kind, "status": "queued", "created": now, "error": None, **extra}
    return job_id


def memory_report():
    try:
        import psutil
        vm = psutil.virtual_memory()
        ram = f"RAM {(vm.total - vm.available) / 1e9:.1f}/{vm.total / 1e9:.1f} GB"
    except Exception:
        ram = "RAM ?"
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        return f"{ram}, VRAM {(total - free) / 1e9:.1f}/{total / 1e9:.1f} GB"
    return ram


# ---------------- Memory guard ----------------
# Colab kills the server when the runtime's RAM (12.7 GB on a T4) runs out. When free RAM gets low, this frees the
# caches this server keeps for speed (they're rebuilt when needed) and hands freed memory back to the system.
RAM_LOW_GB = 3.0        # before a job
RAM_CRITICAL_GB = 2.0   # after a job


def ram_available_gb():
    try:
        with open("/proc/meminfo") as fh:
            for line in fh:
                if line.startswith("MemAvailable:"):
                    return int(line.split()[1]) / 1024 / 1024
    except OSError:
        pass
    return 99.0


def memory_guard(after=False):
    limit = RAM_CRITICAL_GB if after else RAM_LOW_GB
    free = ram_available_gb()
    if free >= limit:
        return
    import ctypes
    import gc as _gc
    freed = []
    try:
        freed = free_caches() or []
    except Exception as e:
        print(f"Memory guard: could not free caches ({e})", flush=True)
    try:
        import comfy.model_management as _mm
        for fn in ("cleanup_models_gc", "cleanup_models"):
            try:
                getattr(_mm, fn)()
            except Exception:
                pass
    except Exception:
        pass
    _gc.collect()
    try:
        import torch as _torch
        _torch.cuda.empty_cache()
    except Exception:
        pass
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)   # give freed memory back to the system
    except Exception:
        pass
    print(f"Memory guard ({free:.1f} GB RAM free {'after' if after else 'before'} a job): freed "
          f"{', '.join(freed) or 'unused memory'} — {ram_available_gb():.1f} GB free now", flush=True)



def free_caches():
    freed = []
    if _COND_CACHE:
        _COND_CACHE.clear(); freed.append("cached prompts")
    if _lora_model_cache:
        _lora_model_cache.clear(); freed.append("LoRA model copies")
    if "pipe" in _depth:
        _depth.clear(); freed.append("depth reader")
    return freed


def run_job(job_id, req):
    job = JOBS[job_id]
    with gpu_lock:
        memory_guard()
        current_job["id"] = job_id
        job["status"] = "running"
        try:
            batch = max(1, min(int(req.batch_size or 1), 8))
            steps = max(1, min(int(req.steps), 60))
            w, h = size16(req.width), size16(req.height)
            seed = int(req.seed) if req.seed is not None and req.seed >= 0 else __import__("random").randint(0, MAX_SEED)
            var_strength = max(0.0, min(float(req.variation_strength or 0), 1.0))
            var_seed = None
            if var_strength > 0:
                var_seed = (int(req.variation_seed) if req.variation_seed is not None and req.variation_seed >= 0
                            else __import__("random").randint(0, MAX_SEED))
            # each picture's seed: seed, seed+1, ... — or, with a variation, the same seed and its own variation seed
            image_seeds = [seed if var_seed is not None else seed + i for i in range(batch)]
            image_var_seeds = [var_seed + i for i in range(batch)] if var_seed is not None else None
            detail_seeds = image_var_seeds or image_seeds
            warns = []
            job.update(warnings=warns)

            # ---- the pictures that come with the request
            face = decode_b64_image((req.face_ref_images or [None])[0]) if (req.face_ref_method or "identity") == "identity" else None
            if req.face_ref_images and (req.face_ref_method or "identity") != "identity":
                warns.append("Krea 2's face reference is Identity Edit — FaceID / PuLID / InstantID are BigLust (SDXL) only.")
            mode = (req.ref_mode or "scene").lower() if req.ref_image else None
            if mode and mode not in REF_MODES:
                raise ValueError(f"Unknown picture mode for Krea 2: {req.ref_mode} (use {', '.join(REF_MODES)})")
            ref_pic = decode_b64_image(req.ref_image) if mode else None
            inpaint = None
            if req.init_image and req.mask_image:
                init = decode_b64_image(req.init_image)
                w, h = size16(init.width), size16(init.height)
                init = init.resize((w, h), Image.Resampling.LANCZOS) if init.size != (w, h) else init
                grow = max(0, min(int(req.mask_grow or 0), 64))
                inpaint = {"init": init, "mask": prepare_mask(decode_b64_image(req.mask_image), (w, h), grow),
                           "feather": max(0, min(int(req.mask_feather or 0), 128)),
                           "denoise": max(0.05, min(float(req.inpaint_denoise or 1.0), 1.0))}
            if mode == "kdepth" and face is not None:
                raise ValueError("Depth and Face reference can't be combined on Krea 2 yet — use the 'Scene' picture mode "
                                 "(puts your character into that picture) or turn one of them off.")
            if mode == "img2img" and inpaint:
                raise ValueError("Img2img and inpainting can't be used together.")
            if req.reference_image or req.reference_images:
                warns.append("Reference Images (Qwen) aren't used by Krea 2 — use Face reference or the Pose / structure picture.")

            # ---- which Edit LoRA, which references (one per task: Identity for a person, General Edit for a picture)
            refs, positions, tool, boost = [], [], None, 1.0
            if face is not None:
                tool = "identity"
                boost = max(0.5, min(float(req.face_ref_strength or 4.0), 12.0))
                if mode in ("scene", "edit"):   # the picture is the scene (aligned), the person sits beside it
                    refs, positions = [ref_pic, face], ["aligned (trained)", "beside (right of target)"]
                else:                            # just the person, restaged freely by the prompt
                    refs, positions = [face], ["beside (right of target)"]
            elif mode in ("scene", "edit"):
                tool, refs, positions = "edit", [ref_pic], ["aligned (trained)"]
            extra = []
            if tool:
                path = tool_file(tool, job)
                extra = [(os.path.basename(path), 1.0)]

            model, lw = model_with_loras(wanted_loras(req.loras, "main"), extra)
            face_base, lw2 = model_with_loras(wanted_loras(req.loras, "face"), extra if face is not None else [])
            warns.extend(x for x in lw + lw2 if x not in warns)
            prompt = req.positive
            if face is not None and "identity" not in prompt.lower():
                prompt = f"{prompt.rstrip(' .')}. Preserve the exact facial identity of this person."
            cfg_on = float(req.cfg) != 1.0
            ref_map = None
            if refs:
                job["phase"] = "reading the reference pictures"
                model = identity_model(model, refs, w, h, boost, positions)
                px = 1024 if face is not None else 768
                positive = grounded(prompt, refs, px)
                negative = grounded("", refs, px) if cfg_on else ConditioningZeroOut.zero_out(positive)[0]
            else:
                positive = encode(prompt)
                negative = ConditioningZeroOut.zero_out(positive)[0] if not cfg_on or not req.negative.strip() else encode(req.negative)
            if mode == "kdepth":
                job["phase"] = "reading the depth"
                model, ref_map = depth_model(model, ref_pic, w, h, max(0.1, min(float(req.ref_strength or 1.0), 1.5)), job)
            job["phase"] = None

            # ---- start latent: empty, img2img or inpaint
            start, denoise = None, 1.0
            if mode == "img2img":
                with torch.inference_mode():
                    start = VAEEncode.encode(vae, pil_to_tensor(fit_cover(ref_pic, w, h)))[0]
                denoise = max(0.05, min(float(req.ref_denoise or 0.6), 1.0))
            elif inpaint:
                with torch.inference_mode():
                    start = VAEEncode.encode(vae, pil_to_tensor(inpaint["init"]))[0]
                    mask_t = torch.from_numpy(np.array(inpaint["mask"]).astype(np.float32) / 255.0)[None, ]
                    start = nodes.NODE_CLASS_MAPPINGS["SetLatentNoiseMask"]().set_mask(start, mask_t)[0]
                denoise = inpaint["denoise"]
            job.update(width=w, height=h)
            print(f"[{job_id}] {w}x{h}, {steps} steps, cfg={req.cfg}, {req.sampler}/{req.scheduler}, batch={batch}, seed={seed}"
                  + (", face: Identity Edit" if face is not None else "") + (f", picture: {mode}" if mode else "")
                  + (", inpaint" if inpaint else "")
                  + (f", LoRAs: {', '.join(f'{n} @ {v}' for n, v in wanted_loras(req.loras, 'main'))}" if req.loras else ""), flush=True)

            # face fix with the face reference: each face crop is redrawn following your face photo
            face_prep = None
            if face is not None:
                face_pos_cache = {}

                def face_prep(size):
                    if "cond" not in face_pos_cache:
                        p = grounded(prompt, [face], 1024)
                        face_pos_cache["cond"] = (p, grounded("", [face], 1024) if cfg_on else ConditioningZeroOut.zero_out(p)[0])
                    m = identity_model(face_base, [face], size, size, boost, ["beside (right of target)"])
                    return (m,) + face_pos_cache["cond"]
            # face fix without a face reference: plain prompt (the edit / scene references aren't used on face crops)
            face_pos = encode(prompt)
            face_neg = ConditioningZeroOut.zero_out(face_pos)[0] if not cfg_on or not req.negative.strip() else encode(req.negative)
            region = {}   # prompts for the hand / eye passes (encoded once)

            def cond(kind):
                if kind not in region:
                    extra_txt = {"hand": ("detailed hand, five fingers, natural hand pose", "extra fingers, missing fingers, fused fingers, deformed hand"),
                                 "eye": ("detailed eyes, sharp iris, catchlights, natural eyelashes", "blurry eyes, asymmetrical eyes, cross-eyed")}[kind]
                    p = encode(f"{req.positive}, {extra_txt[0]}")
                    region[kind] = (p, ConditioningZeroOut.zero_out(p)[0] if not cfg_on or not req.negative.strip()
                                    else encode(f"{req.negative}, {extra_txt[1]}"))
                return region[kind]

            tools_ok = detail_tools_available()
            if (req.face_fix or req.eye_fix or req.hand_fix) and not tools_ok:
                warns.append("Face fix / eye / hand detail aren't installed in this notebook — open it fresh from GitHub and run cell 1 again.")
            if req.face_restore and not face_restore_available():
                warns.append("CodeFormer isn't installed in this notebook — open it fresh from GitHub and run cell 1 again.")
            if req.refiner and (req.refiner_checkpoint or "").strip() not in ("", MODEL_NAME):
                warns.append("Krea 2's refiner always uses the Krea 2 model (other checkpoints are SDXL).")
            if req.refiner and inpaint:
                warns.append("The refiner is skipped when inpainting (it would redraw the whole picture).")
            refine_model = model_with_loras(wanted_loras(req.loras, "main"))[0]
            refine_pos = encode(req.positive)
            refine_neg = ConditioningZeroOut.zero_out(refine_pos)[0] if not cfg_on or not req.negative.strip() else encode(req.negative)

            faces_fixed = hands_fixed = refined = 0
            for i in range(batch):
                with torch.inference_mode(), variation_noise(image_var_seeds[i] if image_var_seeds else None, var_strength):
                    latent = start if start is not None else empty_latent(w, h)
                    samples = KSampler.sample(model, image_seeds[i], steps, float(req.cfg), req.sampler, req.scheduler,
                                              positive, negative, latent, denoise)[0]
                pil = tensor_to_pil(decode_image(samples)[0])
                del samples
                job["phase"] = None

                def tool_step(name, fn):   # a detail tool that fails is skipped (with a warning); the picture is kept
                    nonlocal pil
                    try:
                        pil, n = fn(pil)
                        return n
                    except torch.cuda.OutOfMemoryError:
                        raise
                    except Exception as e:
                        traceback.print_exc()
                        if not any(x.startswith(name) for x in warns):
                            warns.append(f"{name} was skipped: {str(e).splitlines()[0][:200]}")
                        return 0

                if req.face_fix and tools_ok:
                    job["phase"] = "fixing faces"
                    s1 = max(0.1, min(float(req.face_fix_strength or 0.4), 0.8))

                    def face_fix(img):
                        n = 0
                        for k in range(max(1, min(int(req.face_fix_passes or 1), 2))):   # a 2nd pass is lighter
                            img, n = redraw_regions(img, detect_faces(img), face_base, face_pos, face_neg,
                                                    detail_seeds[i] + k, req, s1 if k == 0 else s1 * 0.6, 1.8, prep=face_prep)
                        return img, n
                    faces_fixed += tool_step("Face fix", face_fix)
                if req.eye_fix and tools_ok:
                    job["phase"] = "sharpening the eyes"
                    tool_step("Eye detail", lambda img: redraw_regions(img, eye_boxes(detect_faces(img)), face_base, *cond("eye"), detail_seeds[i],
                                                                       req, max(0.1, min(float(req.eye_fix_strength or 0.3), 0.6)), 1.25))
                if req.hand_fix and tools_ok:
                    job["phase"] = "fixing hands"
                    hands_fixed += tool_step("Hand fix", lambda img: redraw_regions(img, detect_hands(img), model_with_loras(wanted_loras(req.loras, "main"))[0],
                                                                                    *cond("hand"), detail_seeds[i], req,
                                                                                    max(0.1, min(float(req.hand_fix_strength or 0.4), 0.8)), 1.6))
                if inpaint:   # after face fix, so nothing outside the mask can change
                    pil = paste_back(pil, inpaint["init"], inpaint["mask"], inpaint["feather"])
                if req.refiner and not inpaint:
                    refined += tool_step("Refiner", lambda img: (refine(img, req, refine_model, refine_pos, refine_neg, detail_seeds[i], job), 1))
                if req.face_restore and face_restore_available():
                    job["phase"] = "restoring faces"
                    fid = max(0.0, min(float(req.face_restore_fidelity if req.face_restore_fidelity is not None else 0.7), 1.0))
                    tool_step("CodeFormer", lambda img: restore_faces(img, fid, max(0.05, min(float(req.face_restore_blend or 0.4), 1.0)), job))
                job["phase"] = None
                job["images"].append(pil_to_b64(pil))
                job["seeds"].append(image_seeds[i])
                job["done"] = i + 1
                job.update(width=pil.width, height=pil.height)
            if req.face_fix:
                job["faces_fixed"] = faces_fixed
            if req.hand_fix:
                job["hands_fixed"] = hands_fixed
            if image_var_seeds:
                job["variation_seeds"], job["variation_strength"] = image_var_seeds, var_strength
            job.update(face_ref=1 if face is not None else 0, ref_mode=mode, inpainted=bool(inpaint),
                       refined=refined > 0,
                       ref_map=("data:image/png;base64," + pil_to_b64(ref_map)) if ref_map is not None else None)
            job["status"] = "done"
            print(f"[{job_id}] done", flush=True)
        except torch.cuda.OutOfMemoryError as oom:
            msg = str(oom).split("\n")[0]
            del oom   # don't keep the failed attempt's GPU memory alive
            job.update(status="error", error="Out of GPU memory — try a smaller size, batch 1, or an L4 / A100 GPU. " + msg)
        except Exception as e:
            traceback.print_exc()
            job["status"] = "error"
            job["error"] = str(e)
        finally:
            current_job["id"] = None
            job["phase"] = None
            gc.collect()
            torch.cuda.empty_cache()
            print(f"[{job_id}] memory after job: {memory_report()}", flush=True)
            memory_guard(after=True)


def job_payload(job_id):
    job = JOBS[job_id]
    if job.get("kind") == "download":
        return {"job_id": job_id, **{k: v for k, v in job.items() if k != "created"}}
    out = {"job_id": job_id, "status": job["status"], "batch_size": job["batch_size"], "done": job["done"],
           "step": job["step"], "total_steps": job["total_steps"], "width": job.get("width"),
           "height": job.get("height"), "warnings": job.get("warnings", [])}
    if job.get("phase"):
        out["phase"] = job["phase"]
    if job["status"] == "done":
        for k in ("faces_fixed", "hands_fixed", "variation_seeds", "variation_strength", "face_ref", "ref_mode",
                  "inpainted", "refined", "ref_map"):
            if k in job:
                out[k] = job[k]
        out["images"] = job["images"]
        out["image"] = job["images"][0] if job["images"] else None
        out["seeds"] = job["seeds"]
        out["seed"] = job["seeds"][0] if job["seeds"] else None
    if job["status"] == "error":
        out["error"] = job["error"]
    return out


@app.get("/")
def root():
    return {"status": "ok", "model": MODEL_NAME, "mode": "krea2"}


@app.get("/user_models")
def get_user_models():
    gpu = None
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        gpu = {"name": p.name, "vram_gib": round(p.total_memory / 2**30, 1)}
    return {
        "model_type": "krea2",
        "checkpoints": [MODEL_NAME],
        "lora_files": list_files(LORA_DIR),
        "supports_loras": True,
        "max_reference_images": 0,
        "supports_face_fix": detail_tools_available(),
        # the Krea 2 versions of BigLust's tools
        "supports_face_ref": packs_available(),
        "face_ref_methods": ["identity"] if packs_available() else [],
        "max_face_refs": 1,
        "supports_ref_modes": list(REF_MODES) if packs_available() else ["img2img"],
        "supports_inpaint": True,
        "supports_refiner": True,
        "supports_lora_targets": True,
        "supports_lora_tags": True,
        # the BigLust detail tools that work with Krea 2 (PAG, Detail Daemon and Tile are SDXL-only)
        "supports_detail_tools": {"face_fix_passes": detail_tools_available(), "eye_fix": detail_tools_available(),
                                  "hand_fix": detail_tools_available(), "face_restore": face_restore_available(),
                                  "variation_seed": True, "pag": False, "detail_daemon": False, "refiner_tile": False,
                                  "upscalers": upscalers_available()},
        "defaults": KREA_DEFAULTS,
        "gpu": gpu,
        "samplers": ["euler", "euler_ancestral", "res_multistep", "dpmpp_2m", "er_sde", "sa_solver"],
        "schedulers": ["simple", "beta", "normal", "sgm_uniform", "karras"],
    }


@app.post("/generate")
def generate(req: GenerateRequest):
    job_id = new_job("generate", images=[], seeds=[], batch_size=max(1, min(int(req.batch_size or 1), 8)),
                     done=0, step=0, total_steps=0)
    threading.Thread(target=run_job, args=(job_id, req), daemon=True).start()
    return job_payload(job_id)


class ModelDownload(BaseModel):
    url: str
    type: str = "lora"


@app.post("/download_model")
def download_model(req: ModelDownload):
    if req.type != "lora":
        return {"status": "error", "error": "The Krea 2 notebook only downloads LoRAs (the model is fixed)."}
    job_id = new_job("download", type="lora", filename=None, progress=0, done_mb=0, total_mb=None)
    threading.Thread(target=run_download, args=(job_id, req.url, "lora"), daemon=True).start()
    return job_payload(job_id)


@app.get("/job/{job_id}")
def get_job(job_id: str):
    if job_id not in JOBS:
        return {"status": "error", "error": last_crash_message() or "Unknown job (the server restarted)"}
    return job_payload(job_id)


if __name__ == "__main__":
    import uvicorn
    print("Models loaded. Server listening on port 8000.", flush=True)
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")


In [ ]:
# Cell 3: start the generation server as a separate program, plus the Cloudflare tunnel.
# If the server crashes (e.g. runs out of memory) only that program dies: it's restarted automatically
# with the same URL, and this notebook keeps running. Re-running this cell restarts the server with the
# current code from cell 2. To stop everything: Runtime → Restart session.
import os, re, sys, time, subprocess, threading, urllib.request

SERVER_SCRIPT = "/content/krea2_server.py"
SERVER_LOG = "/content/krea2_server.log"
CRASH_FILE = "/content/server_crash.txt"
PORT = 8000
CLOUDFLARED = "/usr/local/bin/cloudflared"


def server_up():
    try:
        return urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=3).status == 200
    except Exception:
        return False


def log_tail(n=15):
    try:
        return "".join(open(SERVER_LOG, errors="replace").readlines()[-n:])
    except OSError:
        return ""


class Supervisor:
    """Runs the server program and restarts it when it crashes (gives up after 4 crashes in 10 minutes)."""

    def __init__(self):
        self.proc, self.thread, self.stopping, self.gave_up, self.crashes = None, None, False, False, []

    def _loop(self):
        while not self.stopping:
            with open(SERVER_LOG, "a") as log:
                log.write(f"\n===== {time.strftime('%H:%M:%S')} starting the server =====\n")
                log.flush()
                self.proc = subprocess.Popen([sys.executable, "-u", SERVER_SCRIPT], cwd="/content/ComfyUI",
                                             stdout=log, stderr=subprocess.STDOUT,
                                             env={**os.environ, "PYTHONUNBUFFERED": "1"},
                                             start_new_session=True)  # stopping a notebook cell must not stop the server
                code = self.proc.wait()
            if self.stopping:
                break
            why = ("ran out of memory and was stopped by the system" if code in (-9, 137)
                   else f"crashed (exit code {code})")
            self.crashes = [t for t in self.crashes if time.time() - t < 600] + [time.time()]
            if len(self.crashes) >= 4:
                msg = (f"The generation server {why} 4 times in 10 minutes, so it was not restarted. "
                       f"Check the log (cell 4), then run cell 3 again.")
                self.gave_up = True
            else:
                msg = (f"The generation server {why} at {time.strftime('%H:%M')}, so that job was lost. "
                       f"It restarts by itself (models reload in 1-3 minutes). Next time try a smaller size, "
                       f"fewer pictures or batch 1.")
            with open(CRASH_FILE, "w") as fh:
                fh.write(msg)
            with open(SERVER_LOG, "a") as log:
                log.write(f"\n!!!!! {msg}\n")
            if self.gave_up:
                break
            time.sleep(3)

    def start(self):
        self.stopping, self.gave_up = False, False
        self.thread = threading.Thread(target=self._loop, daemon=True)
        self.thread.start()

    def stop(self):
        self.stopping = True
        if self.proc and self.proc.poll() is None:
            self.proc.terminate()
            try:
                self.proc.wait(30)
            except subprocess.TimeoutExpired:
                self.proc.kill()
        if self.thread:
            self.thread.join(35)


# (Re)start the server program
if "SUPERVISOR" in globals():
    print("Restarting the server with the current code...")
    try:
        SUPERVISOR.stop()
    except Exception:
        pass
if os.path.exists(CRASH_FILE):
    os.remove(CRASH_FILE)
SUPERVISOR = Supervisor()
SUPERVISOR.start()

print("Loading the models in the server program (1-3 minutes)...")
t0 = time.time()
while not server_up():
    if SUPERVISOR.gave_up or time.time() - t0 > 900:
        print("The server did not start. Last lines of its log:\n" + log_tail(30))
        raise RuntimeError("Server failed to start — see the log above")
    time.sleep(5)
print(f"Server is up after {time.time() - t0:.0f}s.")

# Cloudflare tunnel (started once; the URL stays the same when the server restarts).
# If it stopped (e.g. the runtime hiccuped), a new one is started — with a NEW URL to paste into the website.
if globals().get("TUNNEL_URL") and (globals().get("TUNNEL_PROC") is None or TUNNEL_PROC.poll() is not None):
    print("The old tunnel stopped, so the old URL no longer works — starting a new one (new URL below).")
    TUNNEL_URL = None
if not globals().get("TUNNEL_URL"):
    subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
    if not os.path.exists(CLOUDFLARED):
        subprocess.run(["wget", "-q", "-O", CLOUDFLARED,
                        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
        os.chmod(CLOUDFLARED, 0o755)
    TUNNEL_PROC = subprocess.Popen([CLOUDFLARED, "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                                   start_new_session=True)  # survives stopping a notebook cell
    t0 = time.time()
    while time.time() - t0 < 60 and not globals().get("TUNNEL_URL"):
        line = TUNNEL_PROC.stdout.readline()
        if not line:
            break
        m = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
        if m:
            TUNNEL_URL = m.group(0)
    threading.Thread(target=lambda: [None for _ in TUNNEL_PROC.stdout], daemon=True).start()  # keep the pipe drained

if globals().get("TUNNEL_URL"):
    print("=" * 60)
    print(f"READY: {TUNNEL_URL}")
    print("Paste this into the website's Backend URL field.")
    print("This cell is done; the server keeps running in the background.")
    print("Cell 4 shows its log and memory use. To stop it: Runtime → Restart session.")
    print("=" * 60)
else:
    print("Could not start the Cloudflare tunnel. Run this cell again.")

In [ ]:
# Cell 4 (optional): the server's recent log and memory use. Run it again to refresh.
import subprocess
try:
    import psutil
    vm = psutil.virtual_memory()
    print(f"System RAM: {(vm.total - vm.available) / 1e9:.1f} / {vm.total / 1e9:.1f} GB")
except Exception:
    pass
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    gpu = ""
    print("No GPU in this runtime (Runtime → Change runtime type → L4 GPU).")
if gpu:
    print(f"GPU memory: {gpu}")
sup = globals().get("SUPERVISOR")
if sup is not None:
    running = sup.proc is not None and sup.proc.poll() is None
    print("Server program:", "running" if running else ("stopped — run cell 3" if sup.gave_up else "starting / restarting"))
    print(f"Crashes in the last 10 minutes: {len([t for t in sup.crashes if __import__('time').time() - t < 600])}")
tp = globals().get("TUNNEL_PROC")
if globals().get("TUNNEL_URL"):
    alive = tp is not None and tp.poll() is None
    print(f"Website URL: {TUNNEL_URL}", "(working)" if alive else "— STOPPED: run cell 3 again and paste its new URL into the website")
print("-" * 60)
try:
    print("".join(open("/content/krea2_server.log", errors="replace").readlines()[-40:]))
except OSError:
    print("No log yet — run cell 3 first.")